# Stacked LSTM Model

## Imports, Setup and Data Loading

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error
import pandas as pd

# Configurations

DATASET_ID = 'FD002'          # Change between 'FD001' and 'FD002'
BATCH_SIZE = 256
EPOCHS = 50
INITIAL_LR = 0.003
MAX_GRAD_NORM = 1.0
PATIENCE = 10

SEEDS = [42, 101, 2024, 7, 88, 123, 999, 314, 500, 13]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Data Loading

X_train_arr = np.load(f'../data/processed/X_train_{DATASET_ID}_engine_split.npy')
y_train_arr = np.load(f'../data/processed/y_train_{DATASET_ID}_engine_split.npy')
X_val_arr   = np.load(f'../data/processed/X_val_{DATASET_ID}_engine_split.npy')
y_val_arr   = np.load(f'../data/processed/y_val_{DATASET_ID}_engine_split.npy')

print(f"Train: {X_train_arr.shape}, Val: {X_val_arr.shape}")

class CMAPSSDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

## Stacked LSTM Model 

In [ ]:
class RUL_LSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super(RUL_LSTM, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )
        
    def forward(self, x):
        out, _ = self.lstm(x)
        final_time_step = out[:, -1, :]
        return self.fc(final_time_step).squeeze(-1)

## Generating Metrics

In [ ]:
def compute_cmapss_score(y_true, y_pred):
    diff = y_pred - y_true
    score = np.where(diff < 0, np.exp(-diff / 13.0) - 1, np.exp(diff / 10.0) - 1)
    return np.sum(score)

def evaluate_model(model, data_loader):
    model.eval()
    all_preds, all_trues = [], []
    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            all_preds.extend(outputs.cpu().numpy())
            all_trues.extend(targets.cpu().numpy())
    all_preds = np.array(all_preds)
    all_trues = np.array(all_trues)
    rmse = np.sqrt(mean_squared_error(all_trues, all_preds))
    mae  = mean_absolute_error(all_trues, all_preds)
    nasa = compute_cmapss_score(all_trues, all_preds)
    return rmse, mae, nasa, all_preds

## Multi-Seed Model Training Loop with History Saving

In [ ]:
os.makedirs('../models', exist_ok=True)
os.makedirs('../results/preds', exist_ok=True)
os.makedirs('../results/metrics', exist_ok=True)
os.makedirs('../results/histories', exist_ok=True)

INPUT_SIZE = X_train_arr.shape[2]
results = []

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"SEED {seed} | Dataset {DATASET_ID}")
    print(f"{'='*60}")
    
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    
    # Create DataLoaders
    train_loader = DataLoader(CMAPSSDataset(X_train_arr, y_train_arr),
                              batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(CMAPSSDataset(X_val_arr, y_val_arr),
                              batch_size=BATCH_SIZE, shuffle=False)
    
    # Model and Optimiser
    model = RUL_LSTM(input_size=INPUT_SIZE).to(device)
    criterion = nn.MSELoss()
    optimiser = torch.optim.Adam(model.parameters(), lr=INITIAL_LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimiser, mode='min', factor=0.5, patience=5
    )
    
    best_val_rmse = float('inf')
    patience_counter = 0
    best_state = None

    # History for each seed
    history = {
        "epoch": [],
        "train_rmse": [],
        "val_rmse": [],
        "val_nasa": []
    }
    
    for epoch in range(1, EPOCHS + 1):
        # ---- Train ----
        model.train()
        train_loss = 0.0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimiser.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimiser.step()
            train_loss += loss.item() * inputs.size(0)
        
        train_rmse = np.sqrt(train_loss / len(train_loader.dataset))
        
        # Validation
        val_rmse, val_mae, val_nasa, _ = evaluate_model(model, val_loader)
        scheduler.step(val_rmse)
        
        # Record history
        history["epoch"].append(epoch)
        history["train_rmse"].append(train_rmse)
        history["val_rmse"].append(val_rmse)
        history["val_nasa"].append(val_nasa)
        
        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:02d} | Train RMSE: {train_rmse:.3f} | "
                  f"Val RMSE: {val_rmse:.3f} | MAE: {val_mae:.3f} | NASA: {val_nasa:.1f}")
        
        # Early stopping
        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"Early stopping at epoch {epoch}")
                break
    
    # Load best weights and final evaluation
    model.load_state_dict(best_state)
    final_rmse, final_mae, final_nasa, y_pred = evaluate_model(model, val_loader)
    
    # Save all artefacts
    torch.save(best_state, f'../models/lstm_improved_{DATASET_ID}_seed{seed}.pt')
    np.save(f'../results/preds/lstm_improved_pred_{DATASET_ID}_seed{seed}.npy', y_pred)
    
    # Save history for this seed
    hist_df = pd.DataFrame(history)
    hist_df.to_csv(f'../results/histories/lstm_improved_{DATASET_ID}_seed{seed}.csv', index=False)
    
    # Also save a clean representative history (seed 42) for the appendix
    if seed == 42:
        hist_df.to_csv(f'../results/histories/lstm_improved_{DATASET_ID}.csv', index=False)
        print("Representative history (seed 42) saved for appendix.")
    
    results.append({
        'seed': seed,
        'rmse': final_rmse,
        'mae': final_mae,
        'nasa': final_nasa
    })
    
    print(f"SEED {seed} FINAL → RMSE: {final_rmse:.4f} | MAE: {final_mae:.4f} | NASA: {final_nasa:.2f}")

## Summary

In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv(f'../results/metrics/lstm_improved_{DATASET_ID}_metrics.csv', index=False)

print("\n" + "="*60)
print(f"IMPROVED LSTM – {DATASET_ID} – SUMMARY ACROSS 10 SEEDS")
print("="*60)
print(results_df.describe().round(4))
print("\nMean ± Std:")
print(f"RMSE : {results_df['rmse'].mean():.4f} ± {results_df['rmse'].std():.4f}")
print(f"MAE  : {results_df['mae'].mean():.4f} ± {results_df['mae'].std():.4f}")
print(f"NASA : {results_df['nasa'].mean():.2f} ± {results_df['nasa'].std():.2f}")